<a href="https://colab.research.google.com/github/SaiSanthosh1508/Foundation-Models-From-Scratch/blob/main/Transformer_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import math

class InputEmbeddings(nn.Module):

  def __init__(self, d_model : int, vocab_size : int):
    super().__init__()
    self.d_model = d_model
    self.vocab_size = vocab_size
    self.embedding = nn.Embedding(vocab_size,d_model)

  def forward(self, x):
    return self.embedding(x) * math.sqrt(self.d_model)

In [ ]:
class PositionalEncoding(nn.Module):

  def __init__(self,d_model : int,seq_len : int,dropout : float):
    super().__init__()
    self.d_model = d_model
    self.seq_len = seq_len
    self.dropout = nn.Dropout(dropout)

    # Create a matrix of shape (seq_len,d_model)
    pe = torch.zeros(seq_len,d_model)

    # create position vector
    position = torch.arange(seq_len).reshape(-1,1)

    # create the div term
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

    # Apply sin to even indices and cos to odd indices
    pe[:,0::2] = torch.sin(position * div_term)
    pe[:,1::2] = torch.cos(position * div_term)

    pe = pe.unsqueeze(0)
    self.register_buffer("pe",pe)

  def forward(self,x):
    x = x + self.pe[:, :x.shape[1], :]
    return self.dropout(x)

In [ ]:
class FeedForwardBlock(nn.Module):

  def __init__(self,d_model : int,d_ff : int,dropout : float = 0.1):
    super().__init__()
    self.linear1 = nn.Linear(d_model,d_ff)
    self.dropout = nn.Dropout(dropout)
    self.linear2 = nn.Linear(d_ff,d_model)

  def forward(self, x):
    x = self.linear1(x)
    x = torch.relu(x)
    x = self.dropout(x)
    x = self.linear2(x)
    return x

In [ ]:
import math

class SelfAttentionBlock(nn.Module):

  def __init__(self, d_model : int, d_k : int,d_v : int,dropout : float = 0.1):
    super().__init__()
    self.d_model = d_model

    self.query = nn.Linear(d_model,d_k)
    self.key = nn.Linear(d_model,d_k)
    self.value = nn.Linear(d_model,d_v)
    self.d_k = d_k
    self.d_v = d_v
    self.dropout = nn.Dropout(dropout)

  @staticmethod
  def attention(query, key, value, dropout, mask):
    d_k = query.shape[-1]

    attention_scores = torch.matmul(query,key.transpose(-1,-2)) / math.sqrt(d_k)
    if mask is not None:
      attention_scores = attention_scores.masked_fill(mask==0,-float("inf"))
    attention_scores = attention_scores.softmax(-1)
    if dropout is not None:
      attention_scores = dropout(attention_scores)
    return (attention_scores @ value), attention_scores

  def forward(self, x, mask=None):
    Q = self.query(x)
    K = self.key(x)
    V = self.value(x)
    return self.attention(Q,K,V,self.dropout,mask)

In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads: int, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.h_dim = d_model // num_heads

        self.W_query = nn.Linear(d_model, d_model)
        self.W_key = nn.Linear(d_model, d_model)
        self.W_value = nn.Linear(d_model, d_model)
        self.W_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(q, k, v, dropout, mask=None):
        h_dim = q.shape[-1]
        attn_scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(h_dim)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -float("inf"))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        if dropout is not None:
            attn_weights = dropout(attn_weights)
        return attn_weights @ v

    def forward(self, q_input, k_input, v_input, mask=None):

        q = self.W_query(q_input)
        k = self.W_key(k_input)
        v = self.W_value(v_input)

        b, seq_len_q, _ = q.shape
        b, seq_len_k, _ = k.shape
        b, seq_len_v, _ = v.shape

        # (B, seq_len, d_model) --> (B, num_heads, seq_len, h_dim)
        q = q.view(b, seq_len_q, self.num_heads, self.h_dim).transpose(1, 2)
        k = k.view(b, seq_len_k, self.num_heads, self.h_dim).transpose(1, 2)
        v = v.view(b, seq_len_v, self.num_heads, self.h_dim).transpose(1, 2)

        output = self.attention(q, k, v, self.dropout, mask)

        # (B, num_heads, seq_len_q, h_dim) --> (B, seq_len_q, d_model)
        output = output.transpose(1, 2).contiguous().view(b, seq_len_q, self.d_model)

        # 5. Final output layer
        return self.W_out(output)

In [ ]:
class ResidualConnection(nn.Module):

  def __init__(self,d_model : int, dropout : float):
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    self.norm = nn.LayerNorm(d_model)

  def forward(self,x, sublayer):
    return self.norm(x + self.dropout(sublayer(x)))

In [ ]:
import torch
import torch.nn as nn

class EncoderBlock(nn.Module):

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float):
        super().__init__()

        self.mha = MultiHeadAttention(num_heads, d_model, dropout)
        self.ffn = FeedForwardBlock(d_model, d_ff, dropout)
        self.residuals = nn.ModuleList([
            ResidualConnection(d_model, dropout) for _ in range(2)
        ])

    def forward(self, x, mask=None):


        x = self.residuals[0](x, lambda x: self.mha(x, x, x, mask=mask))

        x = self.residuals[1](x, self.ffn)
        return x

In [ ]:
class Encoder(nn.Module):

    def __init__(self, layers: nn.ModuleList, d_model: int):
        super().__init__()
        self.layers = layers
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)

        return self.norm(x)

In [ ]:
class DecoderBlock(nn.Module):

    def __init__(self, num_heads, d_model, d_ff, dropout):
        super().__init__()

        self.mmha = MultiHeadAttention(num_heads, d_model, dropout)
        self.cross_attention = MultiHeadAttention(num_heads, d_model, dropout)
        self.ffn = FeedForwardBlock(d_model, d_ff, dropout)
        self.residuals = nn.ModuleList([
            ResidualConnection(d_model, dropout) for _ in range(3)
        ])

    def forward(self, x, encoder_output, src_mask, tgt_mask):

        x = self.residuals[0](x, lambda x: self.mmha(x, x, x, mask=tgt_mask))

        x = self.residuals[1](x, lambda x: self.cross_attention(x, encoder_output, encoder_output, mask=src_mask))

        x = self.residuals[2](x, self.ffn)

        return x

In [ ]:
class Decoder(nn.Module):

  def __init__(self,layers : nn.ModuleList,d_model : int):
    super().__init__()
    self.layers = layers
    self.norm = nn.LayerNorm(d_model)

  def forward(self, x, encoder_output,src_mask,tgt_mask):
    for layer in self.layers:
      x = layer(x,encoder_output,src_mask,tgt_mask)
    return self.norm(x)

In [ ]:
class ProjectionLayer(nn.Module):

  def __init__(self,d_model:int,vocab_size:int):
    super().__init__()

    self.proj_layer = nn.Linear(d_model,vocab_size)

  def forward(self, x):
    # (B,seq_len,d_model) --> (B,seq_len,vocab_size)
    return torch.log_softmax(self.proj_layer(x),dim=-1)

In [ ]:
class TransformerBlock(nn.Module):

  def __init__(self, encoder : Encoder, decoder : Decoder, src_embed : InputEmbeddings, tgt_embed : InputEmbeddings, src_pos : PositionalEncoding,tgt_pos : PositionalEncoding, projection_layer : ProjectionLayer):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.src_embed = src_embed
    self.tgt_embed = tgt_embed
    self.src_pos = src_pos
    self.tgt_pos = tgt_pos
    self.projection_layer = projection_layer

  def encode(self, src, src_mask):
    src = self.src_embed(src)
    src = self.src_pos(src)
    return self.encoder(src, src_mask)

  def decode(self, encoder_output, src_mask, tgt, tgt_mask):
    tgt = self.tgt_embed(tgt)
    tgt = self.tgt_pos(tgt)
    return self.decoder(tgt, encoder_output, src_mask, tgt_mask)

  def project(self, x):
    return self.projection_layer(x)

In [ ]:
def build_transformer(src_vocab_size : int, tgt_vocab_size : int,src_seq_len : int,tgt_seq_len : int, d_model : int=512, N : int = 6, h : int = 8, dropout :float = 0.1, d_ff : int = 2048):

  # Create embedding layers
  src_embed = InputEmbeddings(d_model,src_vocab_size)
  tgt_embed = InputEmbeddings(d_model,tgt_vocab_size)

  # Positional Encoding Layers
  src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
  tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout)

  # Create Encoder blocks
  encoder_blocks = []
  for _ in range(N):
    encoder_block = EncoderBlock(d_model, h, d_ff, dropout)
    encoder_blocks.append(encoder_block)

  # Create a list of DecoderBlocks
  decoder_blocks = []
  for _ in range(N):
      decoder_block = DecoderBlock(h, d_model, d_ff, dropout)
      decoder_blocks.append(decoder_block)

  # Pass the *list of blocks* to the Decoder constructor
  decoder = Decoder(nn.ModuleList(decoder_blocks), d_model)
  encoder = Encoder(nn.ModuleList(encoder_blocks),d_model)


  projection_layer = ProjectionLayer(d_model, tgt_vocab_size)

  transformer = TransformerBlock(encoder,decoder,src_embed,tgt_embed,src_pos,tgt_pos,projection_layer)

  # Inititialize the params
  for p in transformer.parameters():
    if p.dim() > 1:
      nn.init.xavier_uniform_(p)

  return transformer

In [ ]:
!pip install -q transformers datasets

## Building Tokenizer

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

def get_all_sentences(ds, lang):
  for item in ds:
    yield item['translation'][lang]


def get_or_build_tokenizer(config, ds, lang):
  tokenizer_path = Path(config['tokenizer_file'].format(lang))
  if not Path.exists(tokenizer_path):
    tokenizer = Tokenizer(WordLevel(unk_token='[UNK]'))
    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordLevelTrainer(special_tokens = ["[UNK]","[PAD]","[SOS]","[EOS]"],min_frequency=2)
    tokenizer.train_from_iterator(get_all_sentences(ds,lang),trainer=trainer)
    tokenizer.save (str(tokenizer_path))
  else:
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
  return tokenizer

In [ ]:
class BilingualDataset(Dataset):

  def __init__(self, ds, tokenizer_src, tokenizer_tgt, src_lang, tgt_lang, seq_len):
    super().__init__()

    self.ds = ds
    self.tokenizer_src = tokenizer_src
    self.tokenizer_tgt = tokenizer_tgt
    self.src_lang = src_lang
    self.tgt_lang = tgt_lang
    self.seq_len = seq_len # Added seq_len initialization

    self.sos_token = torch.tensor([tokenizer_src.token_to_id("[SOS]")],dtype=torch.int64)
    self.eos_token = torch.tensor([tokenizer_src.token_to_id("[EOS]")],dtype=torch.int64)
    self.pad_token = torch.tensor([tokenizer_src.token_to_id("[PAD]")],dtype=torch.int64)

  def __len__(self):
    return len(self.ds)

  def __getitem__(self, index):
    src_target_pair = self.ds[index]
    src_text = src_target_pair['translation'][self.src_lang]
    tgt_text = src_target_pair['translation'][self.tgt_lang]

    enc_input_tokens = self.tokenizer_src.encode(src_text).ids
    dec_input_tokens = self.tokenizer_tgt.encode(tgt_text).ids

    enc_num_padding_tokens = self.seq_len - len(enc_input_tokens) - 2
    dec_num_padding_tokens = self.seq_len - len(dec_input_tokens) - 1

    if enc_num_padding_tokens < 0 or dec_num_padding_tokens < 0:
      raise ValueError("Sentence is too long")

    encoder_input = torch.cat([
        self.sos_token,
        torch.tensor(enc_input_tokens, dtype=torch.int64),
        self.eos_token,
        torch.tensor([self.pad_token] * enc_num_padding_tokens, dtype=torch.int64)
    ])
    decoder_input = torch.cat([
        self.sos_token,
        torch.tensor(dec_input_tokens, dtype=torch.int64),
        torch.tensor([self.pad_token] * dec_num_padding_tokens, dtype=torch.int64)
    ])

    label = torch.cat([
        torch.tensor(dec_input_tokens, dtype=torch.int64),
        self.eos_token,
        torch.tensor([self.pad_token] * dec_num_padding_tokens, dtype=torch.int64)
    ])

    assert encoder_input.size(0) == self.seq_len
    assert decoder_input.size(0) == self.seq_len
    assert label.size(0) == self.seq_len

    return {
        "encoder_input" : encoder_input,
        "decoder_input" : decoder_input,
        "encoder_mask" : (encoder_input != self.pad_token).unsqueeze(0).unsqueeze(0).int(),
        "decoder_mask" : (decoder_input != self.pad_token).unsqueeze(0).unsqueeze(0).int() & causal_mask(decoder_input.size(0)),
        "label" : label,
        "src_text" :src_text,
        "tgt_text" : tgt_text
    }

def causal_mask(size):
  mask = torch.triu(torch.ones(1,size,size),diagonal=1).type(torch.int)
  return mask == 0

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split

def get_ds(config):

  ds_raw = load_dataset('opus_books',f"{config["lang_src"]}-{config["lang_tgt"]}",split="train")

  # Build tokenizer
  tokenizer_src = get_or_build_tokenizer(config,ds_raw,config['lang_src'])
  tokenizer_tgt = get_or_build_tokenizer(config,ds_raw,config['lang_tgt'])

  # Keep 90% for training and 10% for validation
  train_ds_size = int(0.9 * len(ds_raw))
  val_ds_size = len(ds_raw) - train_ds_size

  train_ds_raw,val_ds_raw = random_split(ds_raw, [train_ds_size,val_ds_size])

  train_ds = BilingualDataset(train_ds_raw,tokenizer_src,tokenizer_tgt, config['lang_src'],config['lang_tgt'],config['seq_len'])
  val_ds = BilingualDataset(val_ds_raw,tokenizer_src,tokenizer_tgt, config['lang_src'],config['lang_tgt'],config['seq_len'])

  max_len_src = 0
  max_len_tgt = 0

  for item in ds_raw:
    src_ids = tokenizer_src.encode(item['translation'][config['lang_src']]).ids
    tgt_ids = tokenizer_tgt.encode(item['translation'][config['lang_tgt']]).ids

    max_len_src = max(max_len_src,len(src_ids))
    max_len_tgt = max(max_len_tgt,len(tgt_ids))

  print(f"Max length of source sentence: {max_len_src}")
  print(f"Max length of target sentence: {max_len_tgt}")

  train_dataloader = DataLoader(train_ds,batch_size=config['batch_size'],shuffle=True)
  val_dataloader = DataLoader(val_ds,batch_size=1,shuffle=True)

  return train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt

In [ ]:
def get_model(config, vocab_src_len, vocab_tgt_len):

  model = build_transformer(vocab_src_len,vocab_tgt_len,config['seq_len'],config['seq_len'],config['d_model'])

  return model

In [ ]:
def get_config():
  return  {
      "batch_size" : 8,
      "num_epochs" : 20,
      "lr" : 1e-4,
      "seq_len" : 350,
      "d_model" : 512,
      "lang_src" : "en",
      "lang_tgt" : "it",
      "model_folder" : "weights",
      "model_filename": "tmodel_",
      "preload" : None,
      "tokenizer_file" : "tokenizer_{0}.json",
      "experiment_name" : "runs/tmodel"
  }

def get_weights_file_path(config , epoch : str):
  model_folder = config['model_folder']
  model_basename = config['model_basename']
  model_filename = f"{model_basename}{epoch}.pt"

  return str(Path(".")/ model_folder/ model_filename)



In [ ]:
from  torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

def train_model(config):

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print("Using device", device)

  Path(config['model_folder']).mkdir(parents=True,exist_ok=True)

  train_dataloader,val_dataloader,tokenizer_src,tokenizer_tgt = get_ds(config)
  model = get_model(config,tokenizer_src.get_vocab_size(),tokenizer_tgt.get_vocab_size()).to(device)


  writer = SummaryWriter(config['experiment_name'])

  optimizer = torch.optim.Adam(model.parameters(),lr=config["lr"],eps=1e-9)

  initial_epoch = 0
  global_step = 0

  loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer_src.token_to_id("[PAD]"),label_smoothing=0.1).to(device)

  for epoch in range(initial_epoch, config['num_epochs']):
    model.train()

    batch_iterator = tqdm(train_dataloader, desc=f"Processing epoch{epoch:02d}")

    for batch in batch_iterator:
      encoder_input = batch['encoder_input'].to(device)
      decoder_input = batch['decoder_input'].to(device)
      encoder_mask = batch['encoder_mask'].to(device)
      decoder_mask = batch['decoder_mask'].to(device)
      label = batch['label'].to(device)

      encoder_output = model.encode(encoder_input,encoder_mask)
      decoder_output = model.decode(encoder_output,encoder_mask,decoder_input,decoder_mask)
      proj_output = model.project(decoder_output)

      loss = loss_fn(proj_output.view(-1, tokenizer_tgt.get_vocab_size()), label.view(-1))

      batch_iterator.set_postfix({"loss" : f"{loss.item():6.3f}"})
      writer.add_scalar('train_loss',loss.item(),global_step)
      writer.flush()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad(set_to_none=True)
      global_step += 1

    model_filename = get_weights_file_path(config,f"{epoch:02d}")
    torch.save(model.state_dict(),model_filename)

In [ ]:
config = get_config()
train_model(config)